# 04｜全参数监督微调

130U-FinLLM｜130U 金融大语言模型

任务为针对指定资产的 bearish / neutral / bullish。损失只监督标签位置，更新全部模型参数。检查人工审核与类别分布；不会将普通 positive 标签直接当作 bullish。

每个 Notebook 都包含独立环境入口。Colab 菜单：运行时 → 更改运行时类型 → T4 GPU。点击执行代码前可查看源码；所有输出均为实际执行后产生，仓库不预填训练成绩。

In [ ]:
import os, sys, json, shutil, subprocess, importlib
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # 首次解析并显示提交；复现时替换成完整的 40 位提交 SHA。
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF 不能唯一解析；请填写完整提交 SHA。")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM 已属于其他仓库，请选择新目录。")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("本次固定代码提交：", REVISION)

# 保留 Colab 自带的 CUDA / PyTorch；此处仅安装项目的其他依赖。
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} 与现有环境不兼容：{incompatible}")
    print("tokenizers 兼容性检查通过：", installed)

def check_project_dependencies():
    # Read the refreshed installed metadata, not the previous editable checkout.
    problems = []
    for value in metadata.requires("finllm-130u") or []:
        requirement = Requirement(value)
        if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            problems.append(f"缺少 {requirement.name}")
            continue
        if installed not in requirement.specifier:
            problems.append(f"{requirement.name}=={installed} 不满足 {requirement}")
    if problems:
        raise RuntimeError("项目依赖检查失败：" + "; ".join(problems))
    print("项目已安装依赖检查通过。")

def activate_project_in_kernel():
    # pip installs the editable .pth for NEW Python processes. A notebook kernel
    # that was already running does not process that .pth again automatically.
    source = (PROJECT / "src").resolve()
    expected = source / "finllm"
    if not (expected / "__init__.py").is_file():
        raise RuntimeError(f"缺少项目 Python 源码：{expected}")
    sys.path[:] = [str(source)] + [entry for entry in sys.path if entry != str(source)]
    # Git may have changed code in the same directory since an earlier cell run.
    # Refresh only our modules; never reload Torch or its live CUDA state.
    for name in tuple(sys.modules):
        if name == "finllm" or name.startswith("finllm."):
            del sys.modules[name]
    importlib.invalidate_caches()
    package = importlib.import_module("finllm")
    runtime = importlib.import_module("finllm.runtime")
    for module, filename in ((package, "__init__.py"), (runtime, "runtime.py")):
        if Path(module.__file__).resolve() != expected / filename:
            raise RuntimeError(f"当前 Notebook 导入了错误的项目位置：{module.__file__}")
    if not callable(runtime.restore_seed_checkpoint):
        raise RuntimeError("当前 Notebook 的检查点恢复接口不可用。")
    print("当前 Notebook 内核导入检查通过：", runtime.__file__)

def check_fresh_process_import():
    # Verify the installed editable package separately, without injecting src
    # into PYTHONPATH. Both the current kernel and future CLI processes must work.
    probe = """import json, sys
from pathlib import Path
import finllm
import finllm.runtime as runtime
expected = Path(sys.argv[1]).resolve() / 'finllm'
for module, filename in ((finllm, '__init__.py'), (runtime, 'runtime.py')):
    if Path(module.__file__).resolve() != expected / filename:
        raise RuntimeError('Fresh process imported the wrong project: ' + str(module.__file__))
if not callable(runtime.restore_seed_checkpoint):
    raise RuntimeError('Fresh process checkpoint restore is unavailable')
print(json.dumps({'finllm': str(Path(finllm.__file__).resolve()), 'runtime': str(Path(runtime.__file__).resolve())}))
"""
    result = subprocess.check_output([sys.executable, "-c", probe, str(PROJECT / "src")],
                                     cwd=PROJECT, text=True)
    print("新 Python 进程导入检查通过：", result.strip())

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("需要 PyTorch >= 2.5；请更换较新的 Colab 运行环境。")
tokenizer_requirements = tokenizer_install_requirements()
print("本次 tokenizers 约束：", tokenizer_requirements)
torch_version_before = metadata.version("torch")
torch_file_before = Path(torch.__file__).resolve()
# One resolver transaction replaces the previous finllm metadata AND its
# dependencies. Upgrading tokenizers first leaves old finllm bounds active and
# produces a misleading conflict warning even when the eventual install works.
# The exact installed Torch build is constrained, so pip keeps Colab's CUDA build.
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".",
        "torch==" + torch_version_before, *tokenizer_requirements)
importlib.invalidate_caches()
if metadata.version("torch") != torch_version_before or Path(torch.__file__).resolve() != torch_file_before:
    raise RuntimeError("PyTorch 环境意外变化，请保留输出并停止训练。")
check_tokenizer_compatibility(tokenizer_requirements)
check_project_dependencies()
activate_project_in_kernel()
check_fresh_process_import()
print("Notebook 环境准备完成；PyTorch 保持：", torch_version_before)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("缺少阶段输入，请先准备或从 Drive 恢复：\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"/content 剩余磁盘：{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
# 可选持久化：需要你在 Google 的授权界面完成登录。
MOUNT_DRIVE = False  # 需要备份 / 恢复时设为 True，再执行此单元格。
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Drive 项目目录：", DRIVE_ROOT)
else:
    print("尚未挂载 Drive；/content 是临时磁盘。正式训练前请启用备份。")

In [ ]:
# 新 Notebook 可能分配新的虚拟机；按需恢复前一阶段的文件。
RESTORE_FROM_DRIVE = False
LIVE_STAGES_TO_RESTORE = []  # 中断恢复时填写，例如 ["pretrain"]；会覆盖本地对应阶段。
if RESTORE_FROM_DRIVE:
    if not MOUNT_DRIVE:
        raise RuntimeError("请先把 MOUNT_DRIVE 设为 True 并完成授权。")
    for folder in ("data", "tokenizer", "runs"):
        source = DRIVE_ROOT / folder
        if source.exists():
            finllm("sync", "--source", source, "--destination", PROJECT / folder, "--working-copy")
    for stage in LIVE_STAGES_TO_RESTORE:
        if stage not in ("pretrain", "domain", "sft", "dpo"):
            raise ValueError("未知检查点阶段：" + str(stage))
        source = DRIVE_ROOT / "live_checkpoints" / stage
        if not source.exists():
            raise FileNotFoundError("缺少阶段自动备份：" + str(source))
        finllm("sync", "--source", source, "--destination", PROJECT / "runs" / stage, "--working-copy")
    print("恢复完成。自动检查点只用于匹配原配置、tokenizer 和数据的继续训练；请指定 RESUME_CHECKPOINT。")

In [ ]:
CONFIG = "configs/sft.json"
RESUME_CHECKPOINT = None  # 中断恢复或试跑转正式时，填写 "runs/sft/latest.pt"。
require(CONFIG, 'runs/domain/best.pt', 'tokenizer/english/tokenizer.json', 'data/task/train.jsonl', 'data/task/development.jsonl')
print(Path(CONFIG).read_text(encoding="utf-8"))

In [ ]:
# 正式训练前持久化输入，避免运行时中断后丢失 tokenizer 和数据。
if not MOUNT_DRIVE:
    raise RuntimeError("请先启用 MOUNT_DRIVE 并完成授权，再备份正式训练输入。")
for folder in ("data", "tokenizer"):
    finllm("sync", "--source", PROJECT / folder, "--destination", DRIVE_ROOT / folder)
shutil.copy2(PROJECT / "runtime_revision.txt", DRIVE_ROOT / "runtime_revision.txt")
print("训练输入已备份；训练中会另外自动同步完整检查点。")

In [ ]:
RUN_TRAINING = False  # 阅读配置、检查 GPU 与数据后设为 True。
if not RUN_TRAINING:
    raise RuntimeError("尚未启动正式训练。确认当前阶段与预算后，把 RUN_TRAINING 设为 True。")
if not torch.cuda.is_available():
    raise RuntimeError("正式 24M 训练需要 GPU；请切换到 T4 运行时。")
if not MOUNT_DRIVE:
    raise RuntimeError("正式训练使用 Drive 自动检查点备份，请先完成挂载。")
runtime_config = json.loads(Path(CONFIG).read_text(encoding="utf-8"))
stage = runtime_config["stage"]
runtime_config["backup_dir"] = str(DRIVE_ROOT / "live_checkpoints" / stage)
runtime_path = Path("configs/runtime") / (stage + "_colab.json")
runtime_path.parent.mkdir(parents=True, exist_ok=True)
runtime_path.write_text(json.dumps(runtime_config, indent=2) + "\n", encoding="utf-8")
arguments = ["train", "--config", str(runtime_path)]
if RESUME_CHECKPOINT:
    require(RESUME_CHECKPOINT, str(RESUME_CHECKPOINT) + ".json")
    arguments += ["--resume", RESUME_CHECKPOINT]
finllm(*arguments)

In [ ]:
if MOUNT_DRIVE:
    for folder in ("data", "tokenizer", "runs"):
        source = PROJECT / folder
        if source.exists():
            finllm("sync", "--source", source, "--destination", DRIVE_ROOT / folder)
    shutil.copy2(PROJECT / "runtime_revision.txt", DRIVE_ROOT / "runtime_revision.txt")
    print("备份完成。请保留校验清单，下一运行时按清单恢复。")
else:
    print("成果当前仅在临时运行时中；请挂载 Drive 后重新运行备份单元格。")